Description: <br>
Notebook for evaluating PAKTON and GPT-4o outputs using G-EVAL

Author: Raptopoulos Petros [petrosrapto@gmail.com] <br>
Date  : 2025/07/08

In [8]:
!pip install deepeval
!pip install tqdm


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "your_openai_api_key_here"

In [ ]:
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import GEval
from deepeval.models import GPTModel
from tqdm import tqdm
import json
# Create a custom OpenAI model
custom_model = GPTModel(model="gpt-4o", temperature=0.0)

# Define metric configurations
metric_configs = [
    {
        "name": "Explainability and Reasoning",
        "criteria": (
            "Evaluate whether the report clearly and transparently explains not only the final conclusion, but also the reasoning process and supporting evidence in a step-by-step, understandable manner. "
            "The explanation should guide the reader through the logic in a way that supports comprehension, allowing the reader to see how and why each inference was made. "
            "Assess whether the structure of the reasoning helps build an intuitive understanding of the issue, making the conclusion feel not only justified but inevitable. "
            "The report should avoid unexplained jumps in logic and instead provide a smooth progression that educates the reader along the way, enhancing trust and clarity."
        )
    },
    {
        "name": "Justification with Evidence",
        "criteria": (
            "Determine whether the statements and claims made in the report are explicitly justified with relevant, specific, and clearly cited evidence from the source document(s). "
            "This includes direct quotations, clause references, or document identifiers that can be independently verified. "
            "The justification should be traceable, meaning a reader should be able to locate the original source material and see how it supports the statement. "
            "Also consider whether the report references multiple relevant sources where appropriate, to build a more complete and contextually grounded justification rather than relying on a single source. "
        )
    },
    {
        "name": "Contextual and Legal Understanding",
        "criteria": (
            "Evaluate whether the report demonstrates a deep and accurate understanding of the document itself, the legal terminology it uses, and the broader context in which it is situated. "
            "Assess the correctness of interpretations of clauses, definitions, and legal constructs, including any implicit legal norms or common contractual language. "
            "Additionally, evaluate whether the user's query is correctly understood and interpreted in its full scope — including its legal, contextual, and practical dimensions. "
            "This includes identifying the legal issue(s) being raised, any implied assumptions or concerns, and the broader objectives behind the question."
        )
    },
    {
        "name": "Handling Ambiguity",
        "criteria": "Determine if the report identifies and handles ambiguities in the source material appropriately, such as by discussing multiple interpretations or providing reasoning for one over another."
    },
    {
        "name": "Acknowledgment of Knowledge Gaps",
        "criteria": "Evaluate whether the report explicitly acknowledges when there is insufficient information in the source material to draw a conclusion, and avoids speculation."
    },
    {
        "name": "Conciseness and Precision",
        "criteria": "Assess whether the report communicates clearly and efficiently, avoiding unnecessary repetition or verbosity while still conveying key points thoroughly."
    },
    {
        "name": "Coherence and Organization",
        "criteria": "Check if the report is logically structured, easy to follow, and transitions smoothly between sections or ideas."
    },
    {
        "name": "Relevance and Focus",
        "criteria": "Determine whether the report stays on topic and maintains a clear focus on answering the user's query without introducing off-topic content."
    },
    {
        "name": "Completeness",
        "criteria": (
            "Evaluate if the report covers all important aspects of the query and does not omit any key points that may affect the conclusion or understanding. "
            "The report should demonstrate a broad and holistic view of the problem, approaching the solution from multiple relevant angles or perspectives. "
            "It should avoid overly narrow reasoning and instead include a contextually complete analysis."
        )
    }
]

# Create metrics
metrics = [
    GEval(
        name=config["name"],
        criteria=config["criteria"],
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        model=custom_model
    ) for config in metric_configs
]

# Load data
with open("./results/predictionsPAKTON.json", "r", encoding="utf-8") as f:
    dataPAKTON = json.load(f)

with open("./results/predictionsGPTreasoning.json", "r", encoding="utf-8") as f:
    dataGPT = json.load(f)

# Prepare test cases
test_cases = {"PAKTON": [], "GPT": []}
for index, (pakton_item, gpt_item) in enumerate(zip(dataPAKTON, dataGPT)):
    input_text = pakton_item["system_prompt"]['userQuery'] + '\n' + pakton_item["system_prompt"]['userContext']
    test_cases["PAKTON"].append(LLMTestCase(input=input_text, actual_output=pakton_item["reasoning"], name=f'PAKTON-{index+1}'))
    test_cases["GPT"].append(LLMTestCase(input=input_text, actual_output=gpt_item["reasoning"], name=f'GPT-{index+1}'))

# File to write progressively
output_path = "./results/geval_scores.json"
scores = {}

# Evaluation loop with progress bar
for idx, (pakton_case, gpt_case) in enumerate(tqdm(zip(test_cases["PAKTON"], test_cases["GPT"]), total=len(test_cases["PAKTON"]), desc="Evaluating")):
    case_id = f"test_case_{idx+1}"
    scores[case_id] = {
        "input": pakton_case.input,
        "PAKTON": {"output": pakton_case.actual_output},
        "GPT": {"output": gpt_case.actual_output}
    }

    for model_name, test_case in [("PAKTON", pakton_case), ("GPT", gpt_case)]:
        for metric in metrics:
            metric.measure(test_case)
            scores[case_id][model_name][metric.name] = {
                "score": metric.score,
                "reason": metric.reason
            }

    # Save progress after each test case
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(scores, f, indent=2, ensure_ascii=False)

Evaluating:   0%|          | 0/102 [00:00<?, ?it/s]

/Users/petrosrapto/Desktop/testing/Langchain/venv/lib/python3.11/site-packages/rich/live.py:231: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/Users/petrosrapto/Desktop/testing/Langchain/venv/lib/python3.11/site-packages/rich/live.py:231: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/Users/petrosrapto/Desktop/testing/Langchain/venv/lib/python3.11/site-packages/rich/live.py:231: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/Users/petrosrapto/Desktop/testing/Langchain/venv/lib/python3.11/site-packages/rich/live.py:231: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

...

/Users/petrosrapto/Desktop/testing/Langchain/venv/lib/python3.11/site-packages/rich/live.py:231: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Evaluating:  99%|█████████▉| 101/102 [11:40<00:05,  5.90s/it]

/Users/petrosrapto/Desktop/testing/Langchain/venv/lib/python3.11/site-packages/rich/live.py:231: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/Users/petrosrapto/Desktop/testing/Langchain/venv/lib/python3.11/site-packages/rich/live.py:231: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Evaluating: 100%|██████████| 102/102 [11:46<00:00,  6.93s/it]


# G-Eval Framework Implementation Notes

This notebook implements the G-Eval framework to evaluate and compare the performance of PAKTON and GPT-4o on legal document analysis tasks. The framework scores both models across multiple evaluation criteria including:

1. **Explainability and Reasoning** - Clear explanations of conclusions with step-by-step logic
2. **Justification with Evidence** - Use of specific citations and quotations from source materials
3. **Contextual and Legal Understanding** - Grasp of legal terminology and document context
4. **Handling Ambiguity** - Addressing multiple interpretations appropriately
5. **Acknowledgment of Knowledge Gaps** - Transparency about limitations in available information
6. **Conciseness and Precision** - Clear and efficient communication
7. **Coherence and Organization** - Logical structure and smooth transitions
8. **Relevance and Focus** - Staying on topic and addressing the query directly
9. **Completeness** - Covering all important aspects of the query

The evaluation results are saved to JSON files that can be visualized in the web application.

## Data Processing Pipeline

1. **Test Case Preparation**: Creates test cases from PAKTON and GPT-4o outputs for the same inputs
2. **Metric Evaluation**: Evaluates each model's output using the defined metrics
3. **Score Calculation**: Calculates scores for each criterion and saves them
4. **Data Consolidation**: Combines multiple evaluation runs into a final dataset

The final scores are used in the visualization dashboard to compare performance across different dimensions.

## References

* ChatGPT Enterprise file upload example: https://chatgpt.com/share/e/67f6a8f5-db14-8000-94aa-aa9856695d93
* DeepEval documentation: https://docs.deepeval.com/